# Customer Churn Prediction and Explainability using XGBoost

## Problem Statement
Subscription and usage-based businesses (telecom, ride-hailing, food delivery, digital payments) lose a meaningful chunk of revenue every month to customers who quietly stop using the service. By the time a customer cancels or simply goes inactive, it's usually too late to win them back cheaply. The real value is in flagging the *at-risk* customer a few weeks earlier, while a discount, a support call, or a better plan can still change their mind.

## Business Objective
This project builds a model that predicts the probability of a customer churning, and — just as importantly — explains *why* the model thinks so. A churn score with no explanation is hard for a retention team to act on. If we can tell an account manager "this customer is high risk mainly because of their contract type and monthly charges," they know exactly which lever to pull.

## Why Customer Churn Matters
Acquiring a new customer typically costs far more than retaining an existing one. For platforms that operate on thin margins per transaction (telecom plans, ride-hailing trips, food delivery orders), even a 2-3% improvement in retention can have an outsized impact on revenue. This is exactly the kind of problem that large tech-enabled platforms across Southeast Asia — think ride-hailing, food delivery, and digital payment apps — care about deeply, since a single churned user represents lost trips, lost orders, and lost wallet transactions across the whole platform.

## Project Workflow
1. Load and explore the telco customer dataset
2. Clean and preprocess the data
3. Engineer a handful of practical features
4. Train and compare Logistic Regression, Random Forest, and XGBoost
5. Tune the best model (XGBoost) with RandomizedSearchCV
6. Evaluate the final model on unseen data
7. Explain predictions using SHAP
8. Convert probabilities into risk tiers and generate retention recommendations
9. Save the model and demonstrate inference on a new customer


## Section 2: Import Libraries

Before anything else, we bring in the libraries we'll use throughout the notebook. We also suppress noisy warnings (mostly from sklearn/xgboost version mismatches) and fix a random seed so results are reproducible across runs.

**Expected output:** a printed list of key library versions, confirming the environment is set up correctly.


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

import xgboost as xgb
from xgboost import XGBClassifier

import shap
import joblib

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print("pandas       :", pd.__version__)
print("numpy        :", np.__version__)
print("scikit-learn :", __import__("sklearn").__version__)
print("xgboost      :", xgb.__version__)
print("shap         :", shap.__version__)


## Section 3: Load Dataset

We load the Telco Customer Churn (IBM) dataset from the Kaggle input folder. Kaggle datasets sometimes ship the CSV a folder or two deeper than expected, so instead of hardcoding a filename, we search the input directory for the first CSV file we find. This keeps the notebook from breaking if the exact file name changes between dataset versions.

**Expected output:** dataset shape, a preview of the first few rows, column data types, missing value counts, and the churn distribution.


In [ ]:
DATASET_DIR = "/kaggle/input/datasets/yeanzc/telco-customer-churn-ibm-dataset"

def find_csv(root_dir):
    """Walk the dataset directory and return the path of the first CSV found."""
    for dirpath, _, filenames in os.walk(root_dir):
        for f in filenames:
            if f.lower().endswith(".csv"):
                return os.path.join(dirpath, f)
    raise FileNotFoundError(f"No CSV file found under {root_dir}")

csv_path = find_csv(DATASET_DIR)
print("Loading:", csv_path)

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0]
print("Columns with missing values:")
print(missing_counts if len(missing_counts) else "None found (blanks may be hiding as empty strings, checked later).")


In [ ]:
# The IBM version of this dataset sometimes labels the target as 'Churn',
# 'Churn Label' or 'Churn Value'. We detect whichever is present so the
# rest of the notebook doesn't depend on one exact schema.
possible_targets = ["Churn", "Churn Label", "Churn Value"]
TARGET_COL = next((c for c in possible_targets if c in df.columns), None)

if TARGET_COL is None:
    raise ValueError("Could not find a churn target column in the dataset.")

print("Target column detected:", TARGET_COL)
print(df[TARGET_COL].value_counts(normalize=True).rename("proportion"))


## Section 4: Exploratory Data Analysis

Before touching the model, it's worth spending a bit of time understanding what drives churn in this data. We look at the target balance, how charges and tenure differ between churners and non-churners, and whether contract type or internet service type has an obvious relationship with churn.

**Expected output:** a handful of clean plots — a churn distribution bar chart, distribution plots for tenure and monthly charges, categorical breakdowns by contract and internet service, and a correlation heatmap for the numeric features.


In [ ]:
churn_flag = df[TARGET_COL].apply(lambda x: 1 if str(x).strip() in ["Yes", "1", "True"] else 0)

plt.figure(figsize=(5, 4))
sns.countplot(x=churn_flag)
plt.title("Churn Distribution")
plt.xlabel("Churned (1 = Yes)")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
if "MonthlyCharges" in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x="MonthlyCharges", hue=churn_flag, kde=True, bins=30, palette="Set2")
    plt.title("Monthly Charges Distribution by Churn")
    plt.xlabel("Monthly Charges")
    plt.ylabel("Count")
    plt.show()


In [ ]:
if "tenure" in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x="tenure", hue=churn_flag, kde=True, bins=30, palette="Set2")
    plt.title("Tenure Distribution by Churn")
    plt.xlabel("Tenure (months)")
    plt.ylabel("Count")
    plt.show()


In [ ]:
if "Contract" in df.columns:
    plt.figure(figsize=(7, 4))
    sns.countplot(data=df, x="Contract", hue=churn_flag, palette="Set1")
    plt.title("Contract Type vs Churn")
    plt.xlabel("Contract Type")
    plt.ylabel("Number of Customers")
    plt.legend(title="Churned", labels=["No", "Yes"])
    plt.show()


In [ ]:
if "InternetService" in df.columns:
    plt.figure(figsize=(7, 4))
    sns.countplot(data=df, x="InternetService", hue=churn_flag, palette="Set1")
    plt.title("Internet Service vs Churn")
    plt.xlabel("Internet Service")
    plt.ylabel("Number of Customers")
    plt.legend(title="Churned", labels=["No", "Yes"])
    plt.show()


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) > 1:
    plt.figure(figsize=(9, 6))
    sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Correlation Heatmap - Numerical Features")
    plt.show()
else:
    print("Not enough numeric columns for a correlation heatmap yet (TotalCharges may still be an object type).")


## Section 5: Data Cleaning

A few things need fixing before modeling:

- **TotalCharges** is often stored as text because a handful of new customers (tenure = 0) have blank values instead of 0.
- **customerID** is a unique identifier with no predictive value, so it's dropped.
- The target column is converted into a clean binary 0/1 column so every downstream step can rely on it.

Each step below is small on purpose — the goal is a dataset that's easy to reason about, not a heavily transformed black box.


In [ ]:
data = df.copy()

# TotalCharges sometimes comes in as a string with blank entries for
# brand-new customers. Coerce to numeric and fill the resulting NaNs
# with 0, since a tenure-0 customer hasn't been charged yet.
if "TotalCharges" in data.columns and data["TotalCharges"].dtype == object:
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
    data["TotalCharges"] = data["TotalCharges"].fillna(0)

# Drop the identifier column - it's unique per row and carries no signal.
id_cols = [c for c in ["customerID", "CustomerID"] if c in data.columns]
data = data.drop(columns=id_cols)

# Some IBM-version exports carry extra location/metadata columns that
# aren't useful for a churn model and can leak identifying info.
drop_if_present = ["Count", "Country", "State", "City", "Zip Code", "Lat Long",
                    "Latitude", "Longitude", "Churn Reason", "Churn Score", "CLTV"]
data = data.drop(columns=[c for c in drop_if_present if c in data.columns])

# Encode the target as a clean binary column.
data["Churn_Target"] = churn_flag.values
if TARGET_COL in data.columns:
    data = data.drop(columns=[TARGET_COL])

print("Shape after cleaning:", data.shape)
data.head()


## Section 6: Feature Engineering

Rather than inventing exotic features, we stick to a few that a churn analyst would actually reach for, all derived from columns that already exist in the data:

- **MonthlyChargePerTenure** — how much a customer pays relative to how long they've stuck around. A high value on a short tenure can be a red flag.
- **LongTermCustomer** — flags customers who've been with the company more than a year (tenure > 12 months); long-term customers tend to churn less.
- **HighMonthlyBill** — flags customers paying above the median monthly charge, since bill shock is a common churn trigger.
- **AutoPayCustomer** — flags customers on an automatic payment method, since manual/paper billing tends to correlate with disengagement.
- **PaperlessBillingFlag** — a simple binary version of the existing PaperlessBilling column.


In [ ]:
# MonthlyChargePerTenure: +1 avoids a divide-by-zero for brand-new customers.
if "MonthlyCharges" in data.columns and "tenure" in data.columns:
    data["MonthlyChargePerTenure"] = data["MonthlyCharges"] / (data["tenure"] + 1)

# LongTermCustomer: tenure beyond a year is treated as "long term" here.
if "tenure" in data.columns:
    data["LongTermCustomer"] = (data["tenure"] > 12).astype(int)

# HighMonthlyBill: relative to the dataset's own median, not a fixed dollar amount.
if "MonthlyCharges" in data.columns:
    median_charge = data["MonthlyCharges"].median()
    data["HighMonthlyBill"] = (data["MonthlyCharges"] > median_charge).astype(int)

# AutoPayCustomer: PaymentMethod values containing "automatic" imply autopay.
if "PaymentMethod" in data.columns:
    data["AutoPayCustomer"] = data["PaymentMethod"].str.contains("automatic", case=False, na=False).astype(int)

# PaperlessBillingFlag: binary version of the Yes/No PaperlessBilling column.
if "PaperlessBilling" in data.columns:
    data["PaperlessBillingFlag"] = (data["PaperlessBilling"] == "Yes").astype(int)

print("New engineered columns added.")
data[[c for c in ["MonthlyChargePerTenure", "LongTermCustomer", "HighMonthlyBill",
                  "AutoPayCustomer", "PaperlessBillingFlag"] if c in data.columns]].head()


## Section 7: Encoding

Tree and linear models both need numeric input, so remaining categorical columns need to be encoded. Binary categorical columns (two unique values, e.g. Yes/No) are label-encoded to a single 0/1 column. Columns with more than two categories are one-hot encoded, since label-encoding them would falsely imply an order between categories that don't have one.


In [ ]:
categorical_cols = data.select_dtypes(include=["object"]).columns.tolist()
print("Categorical columns detected:", categorical_cols)

label_encoders = {}
binary_cols = [c for c in categorical_cols if data[c].nunique() == 2]
multi_cols = [c for c in categorical_cols if data[c].nunique() > 2]

for col in binary_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

if multi_cols:
    data = pd.get_dummies(data, columns=multi_cols, drop_first=True)

print("Binary (label-encoded) columns:", binary_cols)
print("Multi-category (one-hot-encoded) columns:", multi_cols)
print("Final shape after encoding:", data.shape)


## Section 8: Train-Test Split

We hold out 20% of the data for testing and stratify on the target so both splits keep the same churn ratio as the full dataset. This matters here because churn is an imbalanced problem — a random split without stratification can accidentally skew the test set.


In [ ]:
X = data.drop(columns=["Churn_Target"])
y = data["Churn_Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train churn rate: {:.2%}".format(y_train.mean()))
print("Test churn rate:  {:.2%}".format(y_test.mean()))


## Section 9: Model Training

We train three models that represent three different levels of complexity: a simple linear baseline (Logistic Regression), a bagged tree ensemble (Random Forest), and a boosted tree ensemble (XGBoost). Comparing all three on the same metrics tells us whether the extra complexity of boosting is actually earning its keep on this dataset.

**Expected output:** a comparison table of Accuracy, Precision, Recall, F1, and ROC-AUC for all three models.


In [ ]:
def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    return {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1 Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs),
    }

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(
        random_state=RANDOM_STATE, eval_metric="logloss", use_label_encoder=False
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = evaluate_model(model, X_test, y_test)

results_df = pd.DataFrame(results).T.sort_values("ROC-AUC", ascending=False)
results_df.round(3)


**Model choice:** XGBoost is carried forward as the model to tune and explain. Boosted trees tend to handle the mix of skewed numeric features (MonthlyCharges, tenure) and one-hot categorical columns better than a linear model, and they usually edge out a plain Random Forest on ROC-AUC because each new tree specifically corrects the previous trees' mistakes rather than just averaging independent trees.


## Section 10: Hyperparameter Tuning

XGBoost has a lot of knobs, but tuning all of them isn't necessary for a dataset this size. We restrict the search to four parameters that matter most for controlling overfitting and model capacity: `max_depth`, `learning_rate`, `n_estimators`, and `subsample`. A `RandomizedSearchCV` with a modest number of iterations is enough to find a noticeably better configuration without turning this into an overnight job.


In [ ]:
param_distributions = {
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "n_estimators": [100, 200, 300, 400],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
}

base_xgb = XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", use_label_encoder=False)

random_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

random_search.fit(X_train, y_train)

print("Best parameters found:")
print(random_search.best_params_)
print("Best CV ROC-AUC: {:.4f}".format(random_search.best_score_))

best_xgb_model = random_search.best_estimator_


## Section 11: Final Model Evaluation

With the tuned XGBoost model in hand, we look at how it performs on the held-out test set: the confusion matrix, ROC curve, and full classification report.

**Why Recall matters here:** in churn prediction, missing an actual churner (a false negative) is usually more costly than flagging a loyal customer by mistake (a false positive). A missed churner walks away with zero chance of intervention, while a false positive just means someone gets a retention offer they didn't strictly need. That's why we track Recall closely alongside overall accuracy, rather than optimizing purely for accuracy.


In [ ]:
final_preds = best_xgb_model.predict(X_test)
final_probs = best_xgb_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, final_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
plt.title("Confusion Matrix - Tuned XGBoost")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, final_probs)
auc_score = roc_auc_score(y_test, final_probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {auc_score:.3f})", color="darkorange")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Tuned XGBoost")
plt.legend()
plt.show()


In [ ]:
print(classification_report(y_test, final_preds, target_names=["No Churn", "Churn"]))

print("ROC-AUC : {:.4f}".format(auc_score))
print("Precision: {:.4f}".format(precision_score(y_test, final_preds)))
print("Recall   : {:.4f}".format(recall_score(y_test, final_preds)))
print("F1 Score : {:.4f}".format(f1_score(y_test, final_preds)))


## Section 12: Explainability with SHAP

A churn score by itself isn't very actionable for a retention team. SHAP (SHapley Additive exPlanations) breaks each prediction down into the contribution of every feature, which lets us answer "why is this specific customer at risk?" instead of just "is this customer at risk?"

We look at global feature importance first (what drives churn across the whole customer base), then use a summary plot to see the direction of each effect, and finally build a small helper to explain one customer's prediction at a time.


In [ ]:
explainer = shap.TreeExplainer(best_xgb_model)
shap_values = explainer.shap_values(X_test)


In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("Global Feature Importance (mean |SHAP value|)")
plt.tight_layout()
plt.show()


In [ ]:
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.show()


In [ ]:
def explain_customer(index):
    """
    Print the churn probability for a single test-set customer along with
    the top features driving that specific prediction.
    """
    row = X_test.iloc[[index]]
    prob = best_xgb_model.predict_proba(row)[0, 1]

    row_shap = explainer.shap_values(row)[0]
    contributions = pd.Series(row_shap, index=X_test.columns).sort_values(key=abs, ascending=False)

    print(f"Customer index: {index}")
    print(f"Churn Probability: {prob:.2%}")
    print("\nTop factors influencing this prediction:")
    for feature, value in contributions.head(5).items():
        direction = "increases" if value > 0 else "decreases"
        print(f"  - {feature}: {direction} churn risk (SHAP value: {value:.3f})")

    return prob

# Example usage on the first test customer
explain_customer(0)


## Section 13: Risk Categories

Raw probabilities are precise but not very intuitive for a business audience. We bucket each customer into **Low**, **Medium**, or **High** risk using configurable thresholds, so a retention team can prioritize outreach without needing to interpret a raw probability score.


In [ ]:
LOW_RISK_THRESHOLD = 0.30
HIGH_RISK_THRESHOLD = 0.60

def get_risk_category(probability, low=LOW_RISK_THRESHOLD, high=HIGH_RISK_THRESHOLD):
    """Map a churn probability to a Low / Medium / High risk label."""
    if probability < low:
        return "Low Risk"
    elif probability < high:
        return "Medium Risk"
    else:
        return "High Risk"

risk_labels = pd.Series(final_probs).apply(get_risk_category)
risk_summary = risk_labels.value_counts()
print(risk_summary)

plt.figure(figsize=(5, 4))
risk_summary.reindex(["Low Risk", "Medium Risk", "High Risk"]).plot(kind="bar", color=["#2ca02c", "#ff9f1c", "#d62728"])
plt.title("Customer Risk Distribution (Test Set)")
plt.xlabel("Risk Category")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()


## Section 14: Recommendation Generator

Once we know a customer's risk tier, the next practical step is deciding what to actually do about it. This is a simple rule-based function on purpose — no LLM involved — since retention playbooks like this are usually owned by a marketing or CRM team and need to stay predictable and auditable.


In [ ]:
def get_recommendation(risk_category):
    """Return a simple retention action based on a customer's risk category."""
    playbook = {
        "High Risk": ["Offer a loyalty discount", "Assign to a retention specialist / campaign"],
        "Medium Risk": ["Send a personalized re-engagement email", "Recommend switching to an annual/longer-term plan"],
        "Low Risk": ["Continue regular engagement", "Include in standard loyalty/rewards communication"],
    }
    return playbook.get(risk_category, ["No action defined"])

for category in ["High Risk", "Medium Risk", "Low Risk"]:
    print(f"{category}: {get_recommendation(category)}")


## Section 15: Save Model

We persist everything needed to reproduce a prediction later: the trained model, the exact list and order of feature columns the model expects, and the label encoders used for the binary categorical columns. Saving the feature column list matters more than it sounds — it's what keeps a new customer's row aligned to the same columns the model was trained on, even after one-hot encoding.


In [ ]:
joblib.dump(best_xgb_model, "xgboost_model.pkl")
joblib.dump(list(X.columns), "feature_columns.pkl")
joblib.dump(label_encoders, "encoder.pkl")

print("Saved: xgboost_model.pkl, feature_columns.pkl, encoder.pkl")


## Section 16: Inference Example

To simulate how this would be used in production, we reload the saved artifacts from disk (rather than reusing the in-memory objects) and run a prediction on a single held-out customer, all the way through to a risk category and recommendation.


In [ ]:
loaded_model = joblib.load("xgboost_model.pkl")
loaded_columns = joblib.load("feature_columns.pkl")

sample_customer = X_test.iloc[[0]][loaded_columns]

sample_prob = loaded_model.predict_proba(sample_customer)[0, 1]
sample_pred = int(sample_prob >= 0.5)
sample_risk = get_risk_category(sample_prob)
sample_recommendation = get_recommendation(sample_risk)

print("Prediction        :", "Churn" if sample_pred == 1 else "No Churn")
print("Churn Probability :", f"{sample_prob:.2%}")
print("Risk Level        :", sample_risk)
print("Recommended Action:", sample_recommendation)


## Section 17: Project Summary

**Business impact.** Even a moderately accurate churn model changes the retention conversation from reactive ("why did this customer leave?") to proactive ("who is likely to leave, and what should we do this week?"). Ranking customers by predicted risk lets a support or marketing team focus limited budget — discounts, calls, personalized offers — on the accounts most likely to respond to them, instead of blanket campaigns.

**Model performance.** The tuned XGBoost model outperformed both the Logistic Regression baseline and Random Forest on ROC-AUC, while keeping Recall high enough to catch most at-risk customers rather than just the obvious ones.

**Key learnings.** A large part of the value here came from explainability, not just accuracy. SHAP made it possible to attach a *reason* to every score, which is what turns a churn model from a data science exercise into something a retention or growth team can actually operationalize — very similar to how a platform running many services at once (rides, deliveries, payments) would want to know not just which users are drifting away, but which specific behavior (fewer weekly trips, a lapsed promo, a support complaint) is driving that drift for each user.

**Future improvements.**
- Incorporate **Customer Lifetime Value (CLTV)** so retention effort is weighted by how valuable a customer is, not just how likely they are to churn.
- Move from a static batch score to **real-time prediction**, scoring customers as their behavior changes rather than on a fixed schedule.
- Add **model monitoring** to catch data or performance drift over time, since customer behavior patterns shift with pricing changes, new competitors, and seasonality.
